In [15]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
print("Installing dependencies...")
!pip install pycryptodome -q
print("Installation complete!")

Installing dependencies...
Installation complete!


In [17]:
import re
import json
import hashlib
import numpy as np
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64
import os
print("All imports successful!")

All imports successful!


Configuration

In [18]:
CONFIG = {
    'max_payload_bits': 4096,
    'ecc_redundancy': 0.3,
    'encryption_enabled': True,
    'master_key': 'steganography_project_2024',
    'max_text_length': 120,
}
print(f"Config loaded: {CONFIG['max_payload_bits']} bits per payload")

Config loaded: 4096 bits per payload


Helper Functions

In [19]:
def text_to_bits(text):
    """Convert text to binary string"""
    return ''.join(format(ord(c), '08b') for c in text)

def bits_to_text(bits):
    """Convert binary string back to text"""
    bits = bits + '0' * (8 - len(bits) % 8 if len(bits) % 8 != 0 else 0)
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        chars.append(chr(int(byte, 2)))
    return ''.join(chars)

def encrypt_text(text, master_key):
    """Encrypt text with AES-256"""
    key = hashlib.sha256(master_key.encode()).digest()
    cipher = AES.new(key, AES.MODE_EAX)
    nonce = cipher.nonce
    ciphertext, tag = cipher.encrypt_and_digest(text.encode('utf-8'))
    encrypted_b64 = base64.b64encode(ciphertext).decode('utf-8')
    encrypted_bits = text_to_bits(encrypted_b64)

    return {
        'encrypted_bits': encrypted_bits,
        'key_id': hashlib.sha256(master_key.encode()).hexdigest()[:16],
        'nonce': base64.b64encode(nonce).decode('utf-8'),
        'tag': base64.b64encode(tag).decode('utf-8'),
        'original_length': len(text)
    }

def decrypt_text(encrypted_bits, master_key, nonce_b64, tag_b64):
    """Decrypt AES-256 encrypted bits"""
    key = hashlib.sha256(master_key.encode()).digest()
    encrypted_b64 = bits_to_text(encrypted_bits).strip('\x00')
    ciphertext = base64.b64decode(encrypted_b64)
    nonce = base64.b64decode(nonce_b64)
    tag = base64.b64decode(tag_b64)
    cipher = AES.new(key, AES.MODE_EAX, nonce=nonce)
    plaintext = cipher.decrypt_and_verify(ciphertext, tag)
    return plaintext.decode('utf-8')

def add_ecc_redundancy(bits, redundancy_factor=0.3):
    """Add redundancy for error correction"""
    repeat_count = max(1, int(1 / (1 - redundancy_factor)))
    return ''.join([bit * repeat_count for bit in bits])

def add_length_header(payload_bits, original_length):
    """Add 32-bit header with payload length"""
    length_bits = format(len(payload_bits), '032b')
    return length_bits + payload_bits

print("Helper functions loaded!")

Helper functions loaded!


Core Preprocessing

In [20]:
def load_and_clean_sentences(file_path):
    """Load and clean sentences"""
    with open(file_path, 'r', encoding='utf-8') as f:
        sentences = [line.strip() for line in f if line.strip()]

    cleaned = []
    for s in sentences:
        s = re.sub(r'[^a-zA-Z0-9 .,?!\'-]', '', s.lower()).strip()
        if len(s) > 10:
            cleaned.append(s)

    print(f"✓ Loaded {len(sentences)} sentences, cleaned to {len(cleaned)}")
    return cleaned

def preprocess_payload(text, config):
    """Complete preprocessing pipeline"""
    # Truncate/pad
    text = text[:config['max_text_length']].strip()

    # Encrypt
    if config['encryption_enabled']:
        encrypted_data = encrypt_text(text, config['master_key'])
        payload_bits = encrypted_data['encrypted_bits']
        metadata = encrypted_data
        metadata['encrypted'] = True  # Add this line
    else:
        payload_bits = text_to_bits(text)
        metadata = {'encrypted': False}

    # Rest of your function unchanged...


    # Add ECC
    if config['ecc_redundancy'] > 0:
        payload_bits = add_ecc_redundancy(payload_bits, config['ecc_redundancy'])
        metadata['ecc_enabled'] = True
        metadata['ecc_redundancy'] = config['ecc_redundancy']

    # Add length header
    original_length = len(payload_bits)
    payload_bits = add_length_header(payload_bits, original_length)
    metadata['total_bits'] = len(payload_bits)

    # Pad or truncate
    max_bits = config['max_payload_bits']
    if len(payload_bits) < max_bits:
        payload_bits += '0' * (max_bits - len(payload_bits))
    else:
        payload_bits = payload_bits[:max_bits]

    # Convert to array
    payload_array = np.array([int(b) for b in payload_bits], dtype=np.float32)

    metadata['original_text'] = text
    metadata['padded_length'] = len(payload_bits)

    return {
        'payload_bits': payload_bits,
        'payload_array': payload_array,
        'metadata': metadata
    }

def process_all_sentences(sentences, config):
    """Process all sentences"""
    processed_payloads = []

    print(f"\nProcessing {len(sentences)} sentences...")
    for i, sentence in enumerate(sentences):
        try:
            payload_data = preprocess_payload(sentence, config)
            processed_payloads.append(payload_data)

            if (i + 1) % 100 == 0:
                print(f"   Processed {i+1}/{len(sentences)}")
        except Exception as e:
            print(f"   ⚠ Error at {i}: {e}")

    print(f"Successfully processed {len(processed_payloads)} payloads")
    return processed_payloads

print("Core functions loaded!")

Core functions loaded!


Save Functions

In [21]:
def save_processed_data(processed_payloads, sentences, output_dir):
    """Save all processed data"""

    # Create output directory if doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Save binary payloads
    binary_only = [p['payload_bits'] for p in processed_payloads]
    with open(f"{output_dir}/binary_payloads.json", "w") as f:
        json.dump(binary_only, f)

    # Save metadata
    metadata_list = [p['metadata'] for p in processed_payloads]
    with open(f"{output_dir}/payload_metadata.json", "w") as f:
        json.dump(metadata_list, f, indent=2)

    # Save cleaned sentences
    with open(f"{output_dir}/cleaned_sentences.txt", "w", encoding="utf-8") as f:
        for s in sentences:
            f.write(s + "\n")

    # Save numpy arrays
    arrays = [p['payload_array'] for p in processed_payloads]
    np.save(f"{output_dir}/payload_arrays.npy", np.array(arrays))

    # Save config
    with open(f"{output_dir}/preprocessing_config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

    print(f"\nAll files saved to: {output_dir}/")
    print(f"binary_payloads.json")
    print(f"payload_metadata.json")
    print(f"cleaned_sentences.txt")
    print(f"payload_arrays.npy")
    print(f"preprocessing_config.json")

print("Save functions loaded!")

Save functions loaded!


Testing

In [22]:
def test_encryption():
    """Test encryption/decryption"""
    print("\nTesting encryption pipeline...")

    test_text = "This is a secret message for steganography!"
    print(f"   Original: {test_text}")

    # Encrypt
    encrypted = encrypt_text(test_text, CONFIG['master_key'])
    print(f"   Encrypted bits: {len(encrypted['encrypted_bits'])} bits")

    # Decrypt
    decrypted = decrypt_text(
        encrypted['encrypted_bits'],
        CONFIG['master_key'],
        encrypted['nonce'],
        encrypted['tag']
    )
    print(f"   Decrypted: {decrypted}")

    match = test_text == decrypted
    print(f"   {'✓' if match else '✗'} Match: {match}")

    return match

print("Test functions loaded!")

Test functions loaded!


Main Execution

In [23]:
def main(file_path, output_dir):
    """Main preprocessing pipeline"""

    print("="*60)
    print("ENHANCED TEXT PREPROCESSING FOR STEGANOGRAPHY")
    print("="*60)

    # Test encryption first
    if not test_encryption():
        print("\nEncryption test failed!")
        return

    # Load and clean
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)
    sentences = load_and_clean_sentences(file_path)
    print(f"\nFirst 3 cleaned sentences:")
    for i, s in enumerate(sentences[:3], 1):
        print(f"   {i}. {s[:60]}...")

    # Process all
    print("\n" + "="*60)
    print("PROCESSING")
    print("="*60)
    processed_payloads = process_all_sentences(sentences, CONFIG)

    # Save
    print("\n" + "="*60)
    print("SAVING")
    print("="*60)
    save_processed_data(processed_payloads, sentences, output_dir)

    # Statistics
    print("\n" + "="*60)
    print("STATISTICS")
    print("="*60)
    avg_bits = np.mean([len(p['payload_bits']) for p in processed_payloads])
    print(f"   Total sentences: {len(sentences)}")
    print(f"   Processed payloads: {len(processed_payloads)}")
    print(f"   Avg bits per payload: {avg_bits:.0f}")
    print(f"   Encryption: {'✓ Enabled (AES-256)' if CONFIG['encryption_enabled'] else '✗ Disabled'}")
    print(f"   ECC redundancy: {CONFIG['ecc_redundancy']*100:.0f}%")
    print(f"   Max capacity: {CONFIG['max_payload_bits']} bits/image")

    print("\n" + "="*60)
    print("PREPROCESSING COMPLETE!")
    print("="*60)
    print("\nNext steps:")
    print("   1. ✓ Text preprocessing DONE")
    print("   2. → Preprocess images (cats & dogs)")
    print("   3. → Implement segmentation (DeepLabV3+)")
    print("   4. → Build U-Net encoder/decoder")

    return processed_payloads

In [24]:
# Set your paths
file_path = "/content/drive/MyDrive/threec/Random English Sentences.txt"
output_dir = "/content/drive/MyDrive/threec/processed"

# Run the complete pipeline
print("Starting preprocessing...\n")
processed = main(file_path, output_dir)

print(f"\nSUCCESS! Processed {len(processed)} payloads")
print(f"Output saved to: {output_dir}")

# Quick Verification

# Load and verify the saved data
print("\n" + "="*60)
print("VERIFYING SAVED DATA")
print("="*60)

# Load arrays
arrays = np.load(f"{output_dir}/payload_arrays.npy")
print(f"✓ Loaded payload_arrays.npy: shape {arrays.shape}")

# Load metadata
with open(f"{output_dir}/payload_metadata.json", "r") as f:
    metadata = json.load(f)
print(f"✓ Loaded payload_metadata.json: {len(metadata)} entries")

# Load config
with open(f"{output_dir}/preprocessing_config.json", "r") as f:
    saved_config = json.load(f)
print(f"✓ Loaded preprocessing_config.json")

# Sample check
print(f"\nSample metadata:")
print(f"Original text: {metadata[0]['original_text'][:50]}...")
print(f"Encrypted: {metadata[0].get('encrypted', 'N/A')}")
print(f"Total bits: {metadata[0]['total_bits']}")

print("\nAll files verified and ready for training!")

Starting preprocessing...

ENHANCED TEXT PREPROCESSING FOR STEGANOGRAPHY

Testing encryption pipeline...
   Original: This is a secret message for steganography!
   Encrypted bits: 480 bits
   Decrypted: This is a secret message for steganography!
   ✓ Match: True

LOADING DATA
✓ Loaded 1000 sentences, cleaned to 999

First 3 cleaned sentences:
   1. i didn't know what to wear to my best friend's funeral....
   2. there used to be a big cherry tree behind my house....
   3. sometimes, books and movies made her feel things she couldn'...

PROCESSING

Processing 999 sentences...
   Processed 100/999
   Processed 200/999
   Processed 300/999
   Processed 400/999
   Processed 500/999
   Processed 600/999
   Processed 700/999
   Processed 800/999
   Processed 900/999
Successfully processed 999 payloads

SAVING

All files saved to: /content/drive/MyDrive/threec/processed/
binary_payloads.json
payload_metadata.json
cleaned_sentences.txt
payload_arrays.npy
preprocessing_config.json

STATISTICS

In [25]:
import random

file_path = "/content/drive/MyDrive/threec/Random English Sentences.txt"

with open(file_path, "r") as f:
    lines = f.readlines()

samples = random.sample(lines, 10)
for s in samples:
    print("-", s.strip())


- I drink less water than him.
- There’s a good chance it’ll rain tomorrow.
- I don't think the house is as big as we hoped.
- My father is in the office.
- Mother doesn't make things up.
- It's been three months and his stab wound hasn't healed yet.
- The gang's all her.
- I want to be married.
- Martha came to the conclusion that shake weights are a great gift for any occasion.
- You shouldn't go swimming after eating a big meal.


In [26]:
# Text attack utilities
import random
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def synonyms(word):
    syns = set()
    for s in wordnet.synsets(word):
        for l in s.lemmas():
            w = l.name().replace('_', ' ')
            if w.lower() != word.lower():
                syns.add(w)
    return list(syns)

def synonym_replacement(sentence, p=0.2):
    tokens = sentence.split()
    for i in range(len(tokens)):
        if random.random() < p:
            s = synonyms(tokens[i])
            if s:
                tokens[i] = random.choice(s)
    return " ".join(tokens)

def random_deletion(sentence, p=0.1):
    tokens = sentence.split()
    if len(tokens) == 1:
        return sentence
    new = [t for t in tokens if random.random() > p]
    if not new:
        return random.choice(tokens)
    return " ".join(new)

def word_shuffle(sentence, max_swap=3):
    tokens = sentence.split()
    n_swaps = min(max_swap, max(1, len(tokens)//3))
    for _ in range(n_swaps):
        i,j = random.sample(range(len(tokens)), 2)
        tokens[i], tokens[j] = tokens[j], tokens[i]
    return " ".join(tokens)

# Example: pick random 6 lines from your text file and show attacks
import random
file_path = "/content/drive/MyDrive/threec/Random English Sentences.txt"
with open(file_path, "r", encoding="utf-8") as f:
    lines = [ln.strip() for ln in f if ln.strip()]

samples = random.sample(lines, min(6, len(lines)))
print("Original | SynonymRepl | RandomDel | Shuffle\n" + "-"*80)
for s in samples:
    print(s)
    print("->", synonym_replacement(s, p=0.25))
    print("->", random_deletion(s, p=0.15))
    print("->", word_shuffle(s))
    print("-"*80)


Original | SynonymRepl | RandomDel | Shuffle
--------------------------------------------------------------------------------
Tom is now looking for a bigger house to live in.
-> Tom cost now looking for a bigger house to live in.
-> Tom is now looking bigger house to live in.
-> live in. now bigger for a looking house to Tom is
--------------------------------------------------------------------------------
You don't need 20 captains.
-> You don't indigence 20 captains.
-> You don't 20 captains.
-> You captains. need 20 don't
--------------------------------------------------------------------------------
You have some schoolwork to do.
-> You have some schoolwork to do.
-> have some schoolwork to do.
-> You have schoolwork do. to some
--------------------------------------------------------------------------------
She told him to grow up.
-> She told him to grow up.
-> She told him to grow
-> She told up. grow to him
-------------------------------------------------------------------